In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('./historical_data.csv')
print(df.head())

   market_id       created_at actual_delivery_time  store_id  \
0        1.0   2/6/2015 22:24       2/6/2015 23:27      1845   
1        2.0  2/10/2015 21:49      2/10/2015 22:56      5477   
2        3.0  1/22/2015 20:39      1/22/2015 21:09      5477   
3        3.0   2/3/2015 21:21       2/3/2015 22:13      5477   
4        3.0   2/15/2015 2:40       2/15/2015 3:20      5477   

  store_primary_category  order_protocol  total_items  subtotal  \
0               american             1.0            4      3441   
1                mexican             2.0            1      1900   
2                    NaN             1.0            1      1900   
3                    NaN             1.0            6      6900   
4                    NaN             1.0            3      3900   

   num_distinct_items  min_item_price  max_item_price  total_onshift_dashers  \
0                   4             557            1239                   33.0   
1                   1            1400            140

In [4]:
import category_encoders as ce
print("category_encoders imported successfully")

category_encoders imported successfully


In [5]:
import category_encoders as ce

df["estimated_store_to_consumer_driving_duration"] = (
    df["estimated_store_to_consumer_driving_duration"]
    .fillna(
        df["estimated_store_to_consumer_driving_duration"].median()
    )
)

df["total_busy_dashers"] = (
    df["total_busy_dashers"]
    .fillna(
        df["total_busy_dashers"].median()
    )
)

df["total_outstanding_orders"] = (
    df["total_outstanding_orders"]
    .fillna(
        df["total_outstanding_orders"].median()
    )
)

df = df.dropna(subset=["estimated_store_to_consumer_driving_duration", "total_outstanding_orders", "total_busy_dashers", "total_onshift_dashers", "created_at", "actual_delivery_time", "store_primary_category", "order_protocol", "market_id"])

df["hour"] = pd.to_datetime(df["created_at"]).dt.hour

df["weekend"] = (
    pd.to_datetime(df["created_at"]).dt.weekday >= 5
).astype(int)

df["interval"] = (
    pd.to_datetime(df["actual_delivery_time"]) -
    pd.to_datetime(df["created_at"])
).dt.total_seconds()

encoder = ce.TargetEncoder(cols=['store_id'])
df['store_id_encoded'] = encoder.fit_transform(
    df[['store_id']],
    df['interval']
).values

df = pd.get_dummies(
    df,
    columns=["market_id"],
    drop_first=True
)

df["store_primary_category"] = df["store_primary_category"].fillna("Unknown")
df = pd.get_dummies(
    df,
    columns=["store_primary_category"],
    drop_first=True
)

df = pd.get_dummies(
    df,
    columns=["order_protocol"],
    drop_first=True
)

X = df.drop(columns=['interval', 'created_at', 'actual_delivery_time'])
y = df['interval']

In [6]:
print("Columns in dataframe:")
print(X.columns.tolist())
print(f"\nDataFrame shape: {X.shape}")
print(f"\nData types:")
print(X.dtypes)

Columns in dataframe:
['store_id', 'total_items', 'subtotal', 'num_distinct_items', 'min_item_price', 'max_item_price', 'total_onshift_dashers', 'total_busy_dashers', 'total_outstanding_orders', 'estimated_order_place_duration', 'estimated_store_to_consumer_driving_duration', 'hour', 'weekend', 'store_id_encoded', 'market_id_2.0', 'market_id_3.0', 'market_id_4.0', 'market_id_5.0', 'market_id_6.0', 'store_primary_category_african', 'store_primary_category_alcohol', 'store_primary_category_alcohol-plus-food', 'store_primary_category_american', 'store_primary_category_argentine', 'store_primary_category_asian', 'store_primary_category_barbecue', 'store_primary_category_belgian', 'store_primary_category_brazilian', 'store_primary_category_breakfast', 'store_primary_category_british', 'store_primary_category_bubble-tea', 'store_primary_category_burger', 'store_primary_category_burmese', 'store_primary_category_cafe', 'store_primary_category_cajun', 'store_primary_category_caribbean', 'store

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [9]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [10]:
predictions = lr.predict(X_test_scaled)

In [11]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, predictions)


print("MAE:", mae)
print('Average', sum(y_test) / len(y_test))

MAE: 658.8073032307054
Average 2862.091914893617


In [12]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_scaled, y_train)

preds = rf.predict(X_test_scaled)

mae = mean_absolute_error(y_test, preds)

print("MAE:", mae)
print('Average', sum(y_test) / len(y_test))

MAE: 645.1292654508612
Average 2862.091914893617
